# 完整组表达特征（外部大文件模式）

仅保留生成全基因表达输入表的步骤，取自已执行的 `16_TF_expression_sensitivity.ipynb`，没有加入旧四分组分析。代码仅适配路径，以下为原始实际运行输出，本次未重新读取完整表达矩阵。完整矩阵作为外部大文件登记；随包已有 `data/figure_inputs/full_gene_expression_summary.tsv`，供五分组离线复算使用。

CP10K 对全部 bin 求均值，包括零表达 bin；检出率分母也为全部 117,867 个 bin。输入 counts 为此前重建的整数，未确认为原始 UMI。

本次只为外部文件按字节计算 SHA256，没有加载这些外部矩阵重新进行评分、相关分析或表达汇总。


In [1]:
from pathlib import Path
import json
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

BASE = Path.cwd()
RESULTS = BASE / "data/analysis_recomputed"
manifest = json.loads((BASE / "manifests/workflow_manifest.json").read_text())
full_regulons = json.loads(Path(manifest["full"]["regulons"]).read_text())
quartiles = ["Q1", "Q2", "Q3", "Q4"]
display(pd.Series({"full_regulons": len(full_regulons), "repeats": len(manifest["runs"]), "full_input": manifest["full"]["input"]}))

full_regulons                                                  260
repeats                                                          5
full_input       /data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
dtype: object

## Full-input expression

For bin b and gene g, CP10K = 10,000 × count(b,g) / total_count(b). Mean CP10K and detection rate use **all bins**. Read the CSR matrix in blocks, without densifying it. The all-gene table also supplies the fixed full-target expression covariates.

In [2]:
with h5py.File(manifest["full"]["input"], "r") as f:
    genes = pd.Index(f["var"][f["var"].attrs["_index"]].asstr()[:], name="gene")
    indptr = f["X/indptr"][:]
    totals = f["obs/total_counts"][:].astype(float)
    n_bins, n_genes = f["X"].attrs["shape"]
    count_sum = np.zeros(n_genes)
    cp10k_sum = np.zeros(n_genes)
    detected = np.zeros(n_genes)
    for start in range(0, n_bins, 4096):
        stop = min(start + 4096, n_bins)
        left, right = indptr[start], indptr[stop]
        values = f["X/data"][left:right]
        indices = f["X/indices"][left:right]
        scales = np.repeat(10000 / totals[start:stop], np.diff(indptr[start:stop + 1]))
        count_sum += np.bincount(indices, weights=values, minlength=n_genes)
        cp10k_sum += np.bincount(indices, weights=values * scales, minlength=n_genes)
        detected += np.bincount(indices, weights=values > 0, minlength=n_genes)

gene_expression = pd.DataFrame({
    "total_counts": count_sum.astype(np.int64),
    "mean_counts": count_sum / n_bins,
    "mean_cp10k": cp10k_sum / n_bins,
    "detection_rate": detected / n_bins,
}, index=genes)
gene_expression.to_csv(RESULTS / "full_gene_expression_summary.tsv", sep="\t")
display(gene_expression.head(), pd.Series({"n_bins": n_bins, "n_genes": n_genes, "total_counts": count_sum.sum(), "sum_mean_cp10k": gene_expression.mean_cp10k.sum()}))

,total_counts,mean_counts,mean_cp10k,detection_rate
gene,,,,
A2M,15095,0.128068,0.591645,0.072429
A2ML1,548,0.004649,0.020926,0.003962
A3GALT2,261,0.002214,0.009115,0.001485
AAAS,4834,0.041012,0.177247,0.031366
AACS,29587,0.251020,1.045895,0.161589


n_bins               117867.0
n_genes               15541.0
total_counts      270918153.0
sum_mean_cp10k        10000.0
dtype: float64